In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

import warnings
warnings.filterwarnings("ignore")

In [ ]:
train = pd.read_csv("Cap_Training_Data_2025.csv")
test = pd.read_csv("Cap_Test_Data_2025.csv")
sample = pd.read_csv("Cap_Sample_Submission_2025.csv")

In [ ]:
print("Training shape:", train.shape)
print("Test shape:", test.shape)
print("Sample shape:", sample.shape)

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

In [ ]:
train.columns

In [ ]:
train.isnull().sum()

In [ ]:
missing = train.isnull().sum()

missing_percentage = (
    missing / len(train) * 100
).sort_values(ascending=False)

missing_percentage

In [ ]:
train.duplicated().sum()

In [ ]:
train = train.drop_duplicates()

In [ ]:
train["Price"].describe()

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(train["Price"], kde=True)

plt.title("Distribution of Car Prices")
plt.xlabel("Price")
plt.ylabel("Frequency")

plt.show()

In [ ]:
numerical_columns = train.select_dtypes(
    include=np.number
).columns

print(numerical_columns)

In [ ]:
train[numerical_columns].hist(
    figsize=(15, 12),
    bins=30
)

plt.tight_layout()
plt.show()

In [ ]:
categorical_columns = train.select_dtypes(
    include="object"
).columns

print(categorical_columns)

In [ ]:
train["Maker"].value_counts().head(20)

In [ ]:
train["fuel_type"].value_counts()

In [ ]:
train["transmission"].value_counts()

In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=train,
    x="Age of car",
    y="Price"
)

plt.title("Car Age vs Price")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=train,
    x="Distance ",
    y="Price"
)

plt.title("Distance vs Price")
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))

correlation = train.select_dtypes(
    include=np.number
).corr()

sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm"
)

plt.title("Feature Correlation")
plt.show()

In [ ]:
numeric_cols = train.select_dtypes(
    include=np.number
).columns

for col in numeric_cols:
    train[col] = train[col].fillna(
        train[col].median()
    )

In [ ]:
categorical_cols = train.select_dtypes(
    include="object"
).columns

for col in categorical_cols:
    train[col] = train[col].fillna(
        train[col].mode()[0]
    )

In [ ]:
train["manufacture_year"] = pd.to_numeric(
    train["manufacture_year"],
    errors="coerce"
)

In [ ]:
X = train.drop("Price", axis=1)

y = train["Price"]

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(X_train.shape)
print(X_valid.shape)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

# Identify categorical columns for one-hot encoding
categorical_features = X.select_dtypes(include='object').columns

# Initialize OneHotEncoder
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit and transform on X_train, then transform X_valid
X_train_encoded = encoder.fit_transform(X_train[categorical_features])
X_valid_encoded = encoder.transform(X_valid[categorical_features])

# Convert encoded features back to DataFrame
X_train_encoded_df = pd.DataFrame(X_train_encoded, columns=encoder.get_feature_names_out(categorical_features), index=X_train.index)
X_valid_encoded_df = pd.DataFrame(X_valid_encoded, columns=encoder.get_feature_names_out(categorical_features), index=X_valid.index)

# Drop original categorical columns and concatenate encoded ones
X_train_processed = X_train.drop(columns=categorical_features).copy()
X_valid_processed = X_valid.drop(columns=categorical_features).copy()

X_train_processed = pd.concat([X_train_processed, X_train_encoded_df], axis=1)
X_valid_processed = pd.concat([X_valid_processed, X_valid_encoded_df], axis=1)

print("Shape of processed training data:", X_train_processed.shape)
print("Shape of processed validation data:", X_valid_processed.shape)

In [141]:
# Initialize and train the RandomForestRegressor model
model = RandomForestRegressor(random_state=42)
model.fit(X_train_processed, y_train)

# Make predictions on the validation set
predictions = model.predict(X_valid_processed)

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(y_valid, predictions))
print(f"Root Mean Squared Error (RMSE): {rmse}")

# Calculate MAE
mae = mean_absolute_error(y_valid, predictions)
print(f"Mean Absolute Error (MAE): {mae}")

# Calculate R2 Score
r2 = r2_score(y_valid, predictions)
print(f"R-squared (R2) Score: {r2}")

KeyboardInterrupt: 

In [ ]:
rmse = np.sqrt(
    mean_squared_error(y_valid, predictions)
)

In [ ]:
import joblib

joblib.dump(
    model,
    "car_price_model.pkl"
)

In [ ]:
# Preprocess the test data using the same steps as the training data

# Impute missing numerical values in test data
numeric_cols_test = test.select_dtypes(include=np.number).columns
for col in numeric_cols_test:
    # Only fill if the column exists in test and was in train numeric columns
    if col in train.select_dtypes(include=np.number).columns:
        test[col] = test[col].fillna(train[col].median())

# Impute missing categorical values in test data
categorical_cols_test = test.select_dtypes(include='object').columns
for col in categorical_cols_test:
    # Only fill if the column exists in test and was in train categorical columns
    if col in train.select_dtypes(include='object').columns:
        test[col] = test[col].fillna(train[col].mode()[0])

# Convert 'manufacture_year' to numeric, coercing errors
test['manufacture_year'] = pd.to_numeric(test['manufacture_year'], errors='coerce')

# Impute any NaN values in 'manufacture_year' that might have resulted from coercion
test['manufacture_year'] = test['manufacture_year'].fillna(train['manufacture_year'].median())

# Apply the same OneHotEncoder fitted on the training data
# Identify categorical columns for one-hot encoding in test data
categorical_features_test = test.select_dtypes(include='object').columns

# Transform the categorical features in the test set using the already fitted encoder
X_test_encoded = encoder.transform(test[categorical_features_test])

# Convert encoded features back to DataFrame
X_test_encoded_df = pd.DataFrame(X_test_encoded, columns=encoder.get_feature_names_out(categorical_features_test), index=test.index)

# Drop original categorical columns and concatenate encoded ones for test data
X_test_processed = test.drop(columns=categorical_features_test).copy()
X_test_processed = pd.concat([X_test_processed, X_test_encoded_df], axis=1)

# Align columns - crucial if test set has different categories or order
X_test_processed = X_test_processed.reindex(columns = X_train_processed.columns, fill_value=0)

print("Shape of processed test data:", X_test_processed.shape)

In [ ]:
test_predictions = model.predict(X_test_processed)
print("Test predictions generated successfully.")
# Display first few predictions
print(test_predictions[:5])

In [ ]:
# Create a submission DataFrame
submission = pd.DataFrame({'ID': sample['ID'], 'Price': test_predictions})

# Display the first few rows of the submission file
print("Submission file head:")
print(submission.head())

In [ ]:
submission = pd.DataFrame({
    "ID": test["ID"],
    "Price": test_predictions
})

In [ ]:
submission.to_csv(
    "predictions.csv",
    index=False
)

In [ ]:
!unzip -o "/content/Indian-Car-Price-Prediction-App.zip" -d "/content/"

In [ ]:
!ls -R /content/Indian-Car-Price-Prediction

In [ ]:
!mkdir -p /content/Indian-Car-Price-Prediction/data

In [ ]:
!cp "/content/Cap_Training_Data_2025.csv" \
"/content/Indian-Car-Price-Prediction/data/"

In [ ]:
!pip install -q streamlit pandas numpy joblib scikit-learn xgboost lightgbm pyngrok

In [ ]:
!mkdir -p /content/Indian-Car-Price-Prediction/models

In [ ]:
!mkdir -p /content/Indian-Car-Price-Prediction/models
!cp /content/car_price_model.pkl /content/Indian-Car-Price-Prediction/models/

In [ ]:
!curl ipv4.icanhazip.com

In [ ]:
!pip install pyngrok -q

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3JHgAROdj4Mc0OGIO3IuNn5G0Vl_6KDR9vMPRvh45xBsLJY2T")

In [ ]:
import os

# Launch Streamlit in the background
os.system("nohup streamlit run /content/Indian-Car-Price-Prediction/app/app.py --server.port 8501 &")

# Create the public ngrok tunnel
public_url = ngrok.connect(8501)
print("Streamlit App URL:", public_url)

In [ ]:
!unzip "/content/Indian-Car-Price-Prediction-App.zip" -d "/content/Indian-Car-Price-Prediction"

In [ ]:
!ls -R /content/Indian-Car-Price-Prediction

In [ ]:
!mkdir -p /content/Indian-Car-Price-Prediction/data

In [ ]:
!cp "/content/Cap_Training_Data_2025.csv" \
"/content/indian-car-price-prediction"

In [ ]:
!pip install -q streamlit pandas numpy joblib scikit-learn xgboost lightgbm pyngrok

In [ ]:
!pip install -q pyngrok

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3JHgAROdj4Mc0OGIO3IuNn5G0Vl_6KDR9vMPRvh45xBsLJY2T")

In [ ]:
!mkdir -p /content/Indian-Car-Price-Prediction/models
!cp /content/car_price_model.pkl /content/Indian-Car-Price-Prediction/models/car_price_pipeline.pkl

In [ ]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor # Added import

# 1. Identify categorical features used during training
categorical_features = ['Maker', 'model', 'Location', 'Owner Type', 'body_type', 'transmission', 'fuel_type']

# 2. Build a preprocessor column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='passthrough'
)

# Initialize the model (e.g., RandomForestRegressor)
model = RandomForestRegressor(random_state=42) # Added model initialization

# 3. Create a unified pipeline containing both preprocessing & model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

# 4. Fit pipeline on raw training features (X_train)
pipeline.fit(X_train, y_train)

# 5. Save the complete pipeline to the Streamlit models folder
!mkdir -p /content/Indian-Car-Price-Prediction/models
joblib.dump(pipeline, '/content/Indian-Car-Price-Prediction/models/car_price_pipeline.pkl')

print("Pipeline created and saved successfully!")

In [ ]:
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# 1. Identify feature groups
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()
numerical_features = X_train.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

# 2. Build preprocessing pipelines
num_transformer = SimpleImputer(strategy='median')
cat_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, numerical_features),
        ('cat', cat_transformer, categorical_features),
    ]
)

# 3. Create the end-to-end Pipeline
full_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', RandomForestRegressor(random_state=42, n_jobs=-1)),
    ]
)

# 4. Fit the complete pipeline on raw training data
full_pipeline.fit(X_train, y_train)

# 5. Save the complete pipeline to the Streamlit app's model path
joblib.dump(
    full_pipeline,
    '/content/Indian-Car-Price-Prediction/models/car_price_pipeline.pkl',
)
print('Pipeline saved successfully!')

In [ ]:
!cp "/content/indian-car-price-prediction" \
"/content/Indian-Car-Price-Prediction/app/app_backup.py"

In [ ]:
!cp "/content/app.py" \
"/content/Indian-Car-Price-Prediction/app/app.py"

In [ ]:
!ls -l /content/Indian-Car-Price-Prediction/app/

In [ ]:
!find /content/Indian-Car-Price-Prediction -maxdepth 3 -type f

In [ ]:
!mkdir -p /content/Indian-Car-Price-Prediction/models
!cp /content/car_price_model.pkl \
/content/Indian-Car-Price-Prediction/models/car_price_model.pkl

In [ ]:
!mkdir -p /content/Indian-Car-Price-Prediction/data

!cp /content/Cap_Training_Data_2025.csv \
/content/Indian-Car-Price-Prediction/data/Cap_Training_Data_2025.csv

In [ ]:
!pip install -q streamlit pyngrok joblib pandas numpy scikit-learn xgboost lightgbm

In [ ]:
!pkill -f streamlit || true

In [ ]:
import subprocess

subprocess.Popen(
    [
        "streamlit",
        "run",
        "/content/Indian-Car-Price-Prediction/app/app.py",
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0",
    ],
    stdout=open("/content/streamlit.log", "w"),
    stderr=subprocess.STDOUT,
)

print("Streamlit started")

In [ ]:
!cat /content/streamlit.log

In [ ]:
!find /content/Indian-Car-Price-Prediction -type f -name "*.py"

In [ ]:
from pyngrok import ngrok

ngrok.kill()

url = ngrok.connect(8501)

print("YOUR STREAMLIT APP:")
print(url)

In [ ]:
!pkill -f streamlit

In [ ]:
!streamlit run /content/Indian-Car-Price-Prediction/app/app.py --server.port 8501 --server.address 0.0.0.0 > /content/streamlit.log 2>&1 &

In [ ]:
!sleep 3
!cat /content/streamlit.log

In [ ]:
!find /content/Indian-Car-Price-Prediction -type f -name "*.py"

In [ ]:
!pkill -f streamlit

In [ ]:
!streamlit run /content/Indian-Car-Price-Prediction/app/app.py --server.port 8501 --server.address 0.0.0.0 > /content/streamlit.log 2>&1 &

In [ ]:
!pip install -q pyngrok

In [ ]:
from pyngrok import ngrok

ngrok.kill()

public_url = ngrok.connect(8501)
print("YOUR APP LINK:")
print(public_url)

In [ ]:
import pandas as pd

pred = pd.read_csv("/content/predictions.csv")

print("Shape:", pred.shape)
print("Columns:", pred.columns.tolist())
print(pred.head())
print(pred.isnull().sum())

In [ ]:
import os

print(os.path.exists(
    "/content/Indian-Car-Price-Prediction/models/car_price_model.pkl"
))

In [ ]:
!mkdir -p /content/Final-Car-Price-Project/app
!mkdir -p /content/Final-Car-Price-Project/models
!mkdir -p /content/Final-Car-Price-Project/data
!mkdir -p /content/Final-Car-Price-Project/outputs

In [ ]:
!cp "/content/Indian-Car-Price-Prediction/app/app.py" \
"/content/Final-Car-Price-Project/app/app.py"

!cp "/content/Indian-Car-Price-Prediction/models/car_price_model.pkl" \
"/content/Final-Car-Price-Project/models/car_price_model.pkl"

!cp "/content/Cap_Training_Data_2025.csv" \
"/content/Final-Car-Price-Project/data/Cap_Training_Data_2025.csv"

!cp "/content/Cap_Test_Data_2025.csv" \
"/content/Final-Car-Price-Project/data/Cap_Test_Data_2025.csv"

!cp "/content/predictions.csv" \
"/content/Final-Car-Price-Project/outputs/predictions.csv"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!mkdir -p "/content/drive/MyDrive/Indian-Car-Price-Prediction/app"
!mkdir -p "/content/drive/MyDrive/Indian-Car-Price-Prediction/models"
!mkdir -p "/content/drive/MyDrive/Indian-Car-Price-Prediction/data"
!mkdir -p "/content/drive/MyDrive/Indian-Car-Price-Prediction/outputs"

In [ ]:
!cp -r "/content/drive/MyDrive/Indian-Car-Price-Prediction" "/content/"

In [ ]:
import os

path = "/content/car_price_model.pkl"
size_mb = os.path.getsize(path) / (1024 * 1024)

print(f"Model size: {size_mb:.2f} MB")

In [ ]:
import joblib
import os

model = joblib.load("/content/car_price_model.pkl")

print("Original size:",
      os.path.getsize("/content/car_price_model.pkl") / (1024*1024),
      "MB")

In [ ]:
joblib.dump(
    model,
    "/content/car_price_model_small.pkl",
    compress=9
)

In [ ]:
size = os.path.getsize("/content/car_price_model_small.pkl") / (1024*1024)

print(f"Compressed model size: {size:.2f} MB")

In [ ]:
import os
import joblib

PROJECT = "/content/Indian-Car-Price-Prediction"

pipeline_path = f"{PROJECT}/models/car_price_pipeline.pkl"
model_path = f"{PROJECT}/models/car_price_model.pkl"

print("=== FILE CHECK ===")
print("Pipeline exists:", os.path.exists(pipeline_path))
print("Model exists:", os.path.exists(model_path))

# Determine which file the Streamlit app will use
if os.path.exists(pipeline_path):
    active_path = pipeline_path
elif os.path.exists(model_path):
    active_path = model_path
else:
    raise FileNotFoundError("No model file found")

print("\n=== ACTIVE MODEL ===")
print(active_path)

model = joblib.load(active_path)

print("\n=== MODEL TYPE ===")
print(type(model))

print("\n=== EXPECTED FEATURES ===")

if hasattr(model, "feature_names_in_"):
    print("Number of features:", len(model.feature_names_in_))
    print(model.feature_names_in_.tolist())
else:
    print("This model does not have feature_names_in_")

print("\n=== PIPELINE STEPS ===")

if hasattr(model, "steps"):
    for name, step in model.steps:
        print(name, "->", type(step))

In [ ]:
# STEP 2 — Train a clean pipeline WITHOUT ID

import os
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

# --------------------------------------------------
# 1. Load training data
# --------------------------------------------------

TRAIN_PATH = "/content/Indian-Car-Price-Prediction/data/Cap_Training_Data_2025.csv"

train_clean = pd.read_csv(TRAIN_PATH)

print("Original shape:", train_clean.shape)
print("Original columns:")
print(train_clean.columns.tolist())


# --------------------------------------------------
# 2. Separate target
# --------------------------------------------------

TARGET = "Price"

X = train_clean.drop(columns=[TARGET]).copy()
y = train_clean[TARGET].copy()


# --------------------------------------------------
# 3. REMOVE ID
# --------------------------------------------------

if "ID" in X.columns:
    X = X.drop(columns=["ID"])

print("\nFeatures after removing ID:")
print(X.columns.tolist())

print("\nNumber of features:", len(X.columns))


# --------------------------------------------------
# 4. Fix column name with trailing space
# --------------------------------------------------

if "Distance " in X.columns:
    X = X.rename(columns={"Distance ": "Distance"})


# --------------------------------------------------
# 5. Make manufacture_year numeric
# --------------------------------------------------

if "manufacture_year" in X.columns:
    X["manufacture_year"] = pd.to_numeric(
        X["manufacture_year"],
        errors="coerce"
    )


# --------------------------------------------------
# 6. Identify numerical and categorical columns
# --------------------------------------------------

numeric_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("\nNumerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)


# --------------------------------------------------
# 7. Numerical preprocessing
# --------------------------------------------------

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)


# --------------------------------------------------
# 8. Categorical preprocessing
# --------------------------------------------------

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)


# --------------------------------------------------
# 9. Combine preprocessing
# --------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)


# --------------------------------------------------
# 10. Create Random Forest
# --------------------------------------------------

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)


# --------------------------------------------------
# 11. Create FINAL pipeline
# --------------------------------------------------

final_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", rf_model)
    ]
)


# --------------------------------------------------
# 12. Train / validation split
# --------------------------------------------------

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


print("\nTraining rows:", X_train.shape[0])
print("Validation rows:", X_valid.shape[0])


# --------------------------------------------------
# 13. Train pipeline
# --------------------------------------------------

print("\nTraining final pipeline...")

final_pipeline.fit(
    X_train,
    y_train
)

print("Training completed.")


# --------------------------------------------------
# 14. Save FINAL pipeline
# --------------------------------------------------

MODEL_DIR = "/content/Indian-Car-Price-Prediction/models"

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)

PIPELINE_PATH = (
    f"{MODEL_DIR}/car_price_pipeline.pkl"
)

joblib.dump(
    final_pipeline,
    PIPELINE_PATH
)

print("\nPipeline saved to:")
print(PIPELINE_PATH)


# --------------------------------------------------
# 15. Verify expected input columns
# --------------------------------------------------

print("\n=== FINAL PIPELINE INPUT FEATURES ===")

print(
    final_pipeline.feature_names_in_.tolist()
)

print(
    "\nNumber of input features:",
    len(final_pipeline.feature_names_in_)
)


# --------------------------------------------------
# 16. IMPORTANT CHECK
# --------------------------------------------------

if "ID" in final_pipeline.feature_names_in_:
    print("\n❌ ERROR: ID is still present!")
else:
    print("\n✅ SUCCESS: ID has been removed from the model.")

In [ ]:
# STEP 3 — Test the new pipeline with one sample car

import joblib
import pandas as pd

PIPELINE_PATH = "/content/Indian-Car-Price-Prediction/models/car_price_pipeline.pkl"

# Load the newly trained pipeline
model = joblib.load(PIPELINE_PATH)

# Create one test car
test_car = pd.DataFrame([{
    "Maker": "audi",
    "model": "coupe",
    "Location": "Ahmedabad",
    "Distance": 50000,
    "Owner Type": "First",
    "manufacture_year": 2020,
    "Age of car": 5,
    "engine_displacement": 1896,
    "engine_power": 91,
    "body_type": "Unknown",
    "Vroom Audit Rating": 6,
    "transmission": "auto",
    "door_count": 4,
    "seat_count": 5,
    "fuel_type": "diesel"
}])

print("=== TEST INPUT ===")
print(test_car)

print("\n=== INPUT COLUMNS ===")
print(test_car.columns.tolist())

print("\n=== MODEL EXPECTS ===")
print(model.feature_names_in_.tolist())

# Make prediction
prediction = model.predict(test_car)

print("\n=== PREDICTION ===")
print(prediction)

print("\nPredicted price:", prediction[0])

In [3]:
!cat /content/app.py

import joblib
import numpy as np
import pandas as pd
import streamlit as st
from pathlib import Path

st.set_page_config(
    page_title="Indian Car Price Predictor",
    page_icon="🚗",
    layout="wide"
)

ROOT = Path(__file__).resolve().parents[1]
MODEL_PATH = ROOT / "models" / "car_price_model.pkl"
PIPELINE_PATH = ROOT / "models" / "car_price_pipeline.pkl"
TRAIN_PATH = ROOT / "data" / "Cap_Training_Data_2025.csv"

TARGET = "Price"
DISTANCE_COL = "Distance "


@st.cache_resource
def load_model():
    # Prefer the pipeline if one exists; otherwise use the existing model.
    if PIPELINE_PATH.exists():
        return joblib.load(PIPELINE_PATH)
    if MODEL_PATH.exists():
        return joblib.load(MODEL_PATH)
    return None


@st.cache_data
def load_training_data():
    if not TRAIN_PATH.exists():
        return None
    return pd.read_csv(TRAIN_PATH)


def build_input_row(train_df, values):
    feature_cols = [c for c in train_df.columns if c != TARGET]
    row = {c: np.nan for c in 

In [4]:
!pkill -f streamlit || true

^C


In [5]:
!streamlit run /content/app.py --server.port 8501 --server.address 0.0.0.0 > /content/streamlit.log 2>&1 &

In [6]:
!cat /content/streamlit.log



2026-10-02 16:40:08.176 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.16.237.5:8501



In [9]:
from pyngrok import ngrok

# Close old tunnels
ngrok.kill()

# Create a fresh tunnel to the running Streamlit app
public_url = ngrok.connect(8501)

print("YOUR NEW STREAMLIT APP:")
print(public_url)

PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://arise-krypton-gangly.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}


In [10]:
from pyngrok import ngrok

# Show tunnels currently known to this Colab session
print(ngrok.get_tunnels())

[]


In [11]:
!ngrok config check
!ngrok version


Valid configuration file at /root/.config/ngrok/ngrok.yml
ngrok version 3.39.11
pyngrok version 8.1.2


In [12]:
!ngrok config check
!ngrok config verify

Valid configuration file at /root/.config/ngrok/ngrok.yml
NAME:
  config - update or migrate ngrok's configuration file

USAGE:
  ngrok config [flags]

DESCRIPTION: 
  The config command gives a quick way to create or update ngrok's configuration
  file. Use 'add-authtoken' or 'add-api-key' to set the corresponding properties.

  Use 'check' to test a configuration file for validity. If you have an old
  configuration file, you can also use 'upgrade' to automatically migrate to the
  latest version.

COMMANDS:
  add-api-key                    save api key to configuration file
  add-authtoken                  save authtoken to configuration file
  add-connect-url                adds the connect URL (connect_url) to configuration file for custom agent ingress
  add-server-addr                alias of add-connect-url
  check                          check configuration file
  edit                           edit configuration file
  upgrade                        auto-upgrade configuratio

In [14]:
from pyngrok import ngrok

# Clean up any local tunnels
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

# Create a new public tunnel
tunnel = ngrok.connect(8501)

print("NEW URL:")
print(tunnel.public_url)

PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://arise-krypton-gangly.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}


In [15]:
from pyngrok import ngrok

# Make sure the old local agent is cleared
ngrok.kill()

# Start Streamlit tunnel
tunnel = ngrok.connect(8501)

print("PUBLIC URL:")
print(tunnel.public_url)

PUBLIC URL:
https://arise-krypton-gangly.ngrok-free.dev


In [18]:
from pyngrok import ngrok

print(ngrok.get_tunnels())

[]


In [22]:
!streamlit run /content/app.py --server.port 8501 --server.address 0.0.0.0 > /content/streamlit.log 2>&1 &


In [24]:
from pyngrok import ngrok

ngrok.kill()

tunnel = ngrok.connect(8501)

print("PUBLIC URL:")
print(tunnel.public_url)

PUBLIC URL:
https://arise-krypton-gangly.ngrok-free.dev


In [27]:
!find /content/Indian-Car-Price-Prediction -maxdepth 3 -type f

/content/Indian-Car-Price-Prediction/Indian-Car-Price-Prediction/app/app.py
/content/Indian-Car-Price-Prediction/Indian-Car-Price-Prediction/README.md
/content/Indian-Car-Price-Prediction/Indian-Car-Price-Prediction/requirements.txt
/content/Indian-Car-Price-Prediction/models/car_price_pipeline.pkl
/content/Indian-Car-Price-Prediction/models/car_price_model.pkl
/content/Indian-Car-Price-Prediction/app/app_backup.py
/content/Indian-Car-Price-Prediction/app/app.py
/content/Indian-Car-Price-Prediction/app/__pycache__/app.cpython-313.pyc
/content/Indian-Car-Price-Prediction/README.md
/content/Indian-Car-Price-Prediction/requirements.txt
/content/Indian-Car-Price-Prediction/data/Cap_Training_Data_2025.csv


In [28]:
!find /content -name "*.ipynb" -type f

/content/drive/MyDrive/Colab Notebooks/code class .ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled1.ipynb
/content/drive/MyDrive/Colab Notebooks/data/Untitled2.ipynb
/content/drive/MyDrive/Colab Notebooks/Copy of data_preprocessing_tools.ipynb
/content/drive/MyDrive/Colab Notebooks/firstproject.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled2.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled3.ipynb
/content/drive/MyDrive/Colab Notebooks/indian__pre_owned_car__price__prediction (5).ipynb
